# Level Zero baseline comparison

This notebook compares **SGD + momentum**, **AdamW**, and **Muon** from the
shared baseline-reference CSV store. It does not reach into individual run
directories.

Each optimizer notebook must first export its completed three-seed result. The
comparison then plots average train/test accuracy, train/test loss,
perplexity, generalization gaps, optimization diagnostics, WeightWatcher
metrics, and ERG gap by epoch. Shading is a **Bollinger-style across-seed
envelope**: mean ± 2 sample standard deviations.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
if (cwd / "configs" / "level0.yaml").is_file():
    EXPERIMENT_ROOT = cwd
elif (cwd.parent / "configs" / "level0.yaml").is_file():
    EXPERIMENT_ROOT = cwd.parent
elif (cwd / "level_0_baseline" / "configs" / "level0.yaml").is_file():
    EXPERIMENT_ROOT = cwd / "level_0_baseline"
else:
    raise FileNotFoundError(
        "Run from the repository or level_0_baseline tree"
    )

sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))

from level0_baseline.analysis import plot_final_test_ci
from level0_baseline.baseline_store import (
    default_baseline_store_root,
    load_common_baseline_store,
    plot_bollinger_summary,
)
from level0_baseline.config import SUPPORTED_OPTIMIZERS

OPTIMIZERS = SUPPORTED_OPTIMIZERS
BASELINE_STORE = default_baseline_store_root()
print(f"baseline store: {BASELINE_STORE}")

## Load and verify the common store

This call fails explicitly if any of the three baseline exports is missing.
The source protocol fingerprints and run directories remain recorded in each
per-optimizer manifest.

In [ ]:
store = load_common_baseline_store(
    BASELINE_STORE,
    require_optimizers=OPTIMIZERS,
)
display(store["manifest"])

trajectory_runs = store["trajectory_runs"]
epoch_runs = store["epoch_runs"]
spectral_runs = store["spectral_runs"]
terminal_test_runs = store["terminal_test_runs"]

trajectory_summary = store["trajectory_summary"]
epoch_summary = store["epoch_summary"]
spectral_summary = store["spectral_summary"]
terminal_test_summary = store["terminal_test_summary"]

display(
    epoch_runs[
        [
            "optimizer_label",
            "seed",
            "nominal_epoch",
            "train_loss",
            "test_loss",
            "train_accuracy",
            "test_accuracy",
            "test_perplexity",
        ]
    ].sort_values(
        ["optimizer_label", "seed", "nominal_epoch"]
    )
)

## Average train and test loss by epoch

All bands are computed from the same three seeds on the same nominal epoch
grid. The exported `lower` and `upper` columns are mean ± 2 sample SD.

In [ ]:
for metric in [
    "train_loss",
    "val_loss",
    "test_loss",
]:
    plot_bollinger_summary(
        epoch_summary,
        metric=metric,
        x="nominal_epoch",
        optimizers=OPTIMIZERS,
        title=f"{metric} by epoch: mean ± 2 SD",
    )
    plt.show()

## Average train and test accuracy by epoch

In [ ]:
for metric in [
    "train_accuracy",
    "val_accuracy",
    "test_accuracy",
]:
    plot_bollinger_summary(
        epoch_summary,
        metric=metric,
        x="nominal_epoch",
        optimizers=OPTIMIZERS,
        title=f"{metric} by epoch: mean ± 2 SD",
    )
    plt.show()

## Average perplexity by epoch

In [ ]:
for metric in [
    "train_perplexity",
    "val_perplexity",
    "test_perplexity",
]:
    plot_bollinger_summary(
        epoch_summary,
        metric=metric,
        x="nominal_epoch",
        optimizers=OPTIMIZERS,
        title=f"{metric} by epoch: mean ± 2 SD",
    )
    plt.show()

## Generalization and optimization-quality metrics

In [ ]:
for metric in [
    "val_generalization_gap",
    "test_generalization_gap",
    "grad_norm_pre_clip",
    "update_to_weight_ratio",
    "weight_norm",
    "tokens_per_sec",
]:
    plot_bollinger_summary(
        epoch_summary,
        metric=metric,
        x="nominal_epoch",
        optimizers=OPTIMIZERS,
        title=f"{metric} by epoch: mean ± 2 SD",
    )
    plt.show()

## Full-resolution train/validation trajectories

These curves use every normal evaluation point rather than only the five
integer-epoch checkpoints.

In [ ]:
for metric in [
    "train_loss",
    "val_loss",
    "train_accuracy",
    "val_accuracy",
    "val_generalization_gap",
    "grad_norm_pre_clip",
    "update_to_weight_ratio",
]:
    plot_bollinger_summary(
        trajectory_summary,
        metric=metric,
        x="epoch",
        optimizers=OPTIMIZERS,
        title=f"Full-resolution {metric}: mean ± 2 SD",
    )
    plt.show()

## WeightWatcher alpha, ERG gap, and fit-quality overlays

In [ ]:
for metric in [
    "alpha_median",
    "ERG_gap_median",
    "D_median",
    "stable_rank_median",
    "mp_softrank_median",
]:
    plot_bollinger_summary(
        spectral_summary,
        metric=metric,
        x="epoch",
        optimizers=OPTIMIZERS,
        title=f"{metric}: mean ± 2 SD",
    )
    if metric == "alpha_median":
        plt.axhline(2.0, linestyle="--", linewidth=1.0)
    if metric == "ERG_gap_median":
        plt.axhline(0.0, linestyle="--", linewidth=1.0)
    plt.show()

## Final and validation-selected test summaries

These bars use two-sided 95% Student-t intervals across the three seeds. They
are distinct from the ±2-SD trajectory envelopes.

In [ ]:
display(
    terminal_test_summary[
        [
            "optimizer_label",
            "checkpoint",
            "metric",
            "n",
            "mean",
            "sd",
            "ci95_half_width",
            "ci95_lower",
            "ci95_upper",
        ]
    ]
)

for checkpoint in ["final", "validation_selected"]:
    for metric in [
        "test_loss",
        "test_perplexity",
        "test_accuracy",
    ]:
        plot_final_test_ci(
            terminal_test_summary,
            metric=metric,
            checkpoint=checkpoint,
            optimizers=OPTIMIZERS,
        )
        plt.show()

## Compact baseline table

This table is suitable for carrying into later experiment notebooks. It uses
the final integer-epoch row and keeps the mean, sample SD, and Bollinger
envelope for each optimizer and metric.

In [ ]:
final_epoch = epoch_summary[
    epoch_summary["nominal_epoch"]
    == epoch_summary["nominal_epoch"].max()
]
baseline_table = final_epoch[
    final_epoch["metric"].isin(
        [
            "train_loss",
            "test_loss",
            "train_accuracy",
            "test_accuracy",
            "test_perplexity",
            "test_generalization_gap",
        ]
    )
][
    [
        "optimizer_label",
        "metric",
        "n",
        "mean",
        "sd",
        "lower",
        "upper",
    ]
].sort_values(["metric", "optimizer_label"])

display(baseline_table)

The CSV files under `baseline_reference/summaries/` are the stable
baseline inputs for future experiments. Later notebooks should join on
`optimizer`, `metric`, and `epoch` rather than re-deriving baseline statistics
from ad hoc paths.